# Ghana Indigenous Intel Challenge — Rainfall Classification

**Competition:** [Ghana Indigenous Intel Challenge](https://zindi.africa/competitions/ghana-indigenous-intel-challenge) (Zindi)

**Objective:** Build a classification model that predicts rainfall type (heavy, moderate, small, or none) expected in the next 12-24 hours, based solely on indigenous ecological indicators submitted by trained farmers in Ghana.

**Approach:** Sklearn pipeline with Random Forest classifier, using time-based and categorical features extracted from farmer submissions. Preprocessing handles missing values (median imputation for numeric, most-frequent for categorical) and one-hot encodes categorical features.

**Result:** Leaderboard score of **0.9587** (F1 macro), CV macro F1 of **0.9852 +/- 0.0018**.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

In [ ]:
# Load data
TRAIN_PATH = "data/train.csv"
TEST_PATH = "data/test.csv"
SAMPLE_SUB_PATH = "data/SampleSubmission.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

## Exploratory Data Analysis

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
# Target distribution - dataset is heavily imbalanced toward NORAIN
cls_pct = train["Target"].value_counts(normalize=True).mul(100).round(2)
print("Class distribution (%):\n", cls_pct)

plt.figure(figsize=(8, 4))
cls_pct.sort_values().plot(kind="bar")
plt.title("Target distribution in training data (%)")
plt.ylabel("%")
plt.tight_layout()
plt.show()

## Feature Engineering

In [ ]:
def parse_time_features(df, time_col="prediction_time"):
    """Extract hour, day-of-week, and date from the prediction timestamp."""
    df = df.copy()
    if time_col in df.columns:
        dt = pd.to_datetime(df[time_col].astype(str), dayfirst=True, errors="coerce")
        df["pred_hour"] = dt.dt.hour
        df["pred_dow"] = dt.dt.dayofweek
        df["pred_date"] = dt.dt.date.astype("str")
    return df

train = parse_time_features(train)
test = parse_time_features(test)

In [ ]:
TARGET = "Target"
ID_COL = "ID"

# All columns except target are candidate features
feature_cols = [c for c in train.columns if c != TARGET]

# Drop columns that are all-NaN or redundant after time feature extraction
drop_cols = ["prediction_time", "time_observed", "indicator_description", "confidence", "predicted_intensity"]
feature_cols = [c for c in feature_cols if c not in drop_cols]

# Separate numeric and categorical features (excluding ID)
numeric_cols = [c for c in feature_cols if c != ID_COL and pd.api.types.is_numeric_dtype(train[c])]
cat_cols = [c for c in feature_cols if c != ID_COL and not pd.api.types.is_numeric_dtype(train[c])]

print("Numeric features:", numeric_cols)
print("Categorical features:", cat_cols)

## Model Pipeline

In [ ]:
# Preprocessing: median imputation for numeric, most-frequent + OHE for categorical
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, cat_cols),
])

# Full pipeline: preprocessing + Random Forest classifier
model = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)

pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model),
])

## Cross-Validation

In [ ]:
X = train[feature_cols].drop(columns=[ID_COL], errors="ignore")
y = train[TARGET]

# 5-fold stratified CV to preserve class balance across splits
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_macro_scores = cross_val_score(pipe, X, y, cv=skf, scoring="f1_macro", n_jobs=-1)
acc_scores = cross_val_score(pipe, X, y, cv=skf, scoring="accuracy", n_jobs=-1)

print(f"CV Macro F1: {f1_macro_scores.mean():.4f} +/- {f1_macro_scores.std():.4f}")
print(f"CV Accuracy: {acc_scores.mean():.4f} +/- {acc_scores.std():.4f}")

In [ ]:
# Fit on full training data and inspect predictions
pipe.fit(X, y)
y_pred = pipe.predict(X)

print("Confusion Matrix:")
print(confusion_matrix(y, y_pred))
print("\nClassification Report:")
print(classification_report(y, y_pred))

## Generate Submission

In [ ]:
def conform_to_sample(sample_df, pred_df, id_col="ID"):
    """Align predictions to match the row order and column format of the sample submission."""
    sample_cols = list(sample_df.columns)
    assert id_col in sample_cols, f"'{id_col}' must be a column in SampleSubmission"
    target_cols = [c for c in sample_cols if c != id_col]

    if len(target_cols) == 0:
        raise ValueError("SampleSubmission must contain at least one target column besides the id.")

    merged = sample_df[[id_col]].merge(pred_df, on=id_col, how="left")

    for tcol in target_cols:
        if tcol in pred_df.columns:
            merged[tcol] = merged[tcol]
        else:
            pred_only = [c for c in pred_df.columns if c != id_col]
            if len(pred_only) == 1:
                merged[tcol] = merged[pred_only[0]]
            else:
                raise ValueError(f"Cannot map predictions to sample target column '{tcol}'.")

    return merged[sample_cols]

In [ ]:
# Train on full data and predict test set
pipe.fit(X, y)
X_test = test[feature_cols].drop(columns=[ID_COL], errors="ignore")
test_pred = pipe.predict(X_test)

# Create submission DataFrame and align to sample format
pred_df = pd.DataFrame({ID_COL: test[ID_COL].values, "rain_type": test_pred})
submission = conform_to_sample(sample_sub, pred_df, id_col=ID_COL)

submission.head()

In [ ]:
# Save submission
save_path = "submission.csv"
submission.to_csv(save_path, index=False)
print(f"Saved submission to: {save_path}")